# Gradient Boosting — Grupo 8

**Pregunta:** ¿puede un ensamble de árboles con **Gradient Boosting** capturar relaciones no lineales e interacciones entre variables para superar de forma consistente tanto al baseline logístico como al árbol de decisión simple?

Para asegurar una comparación metodológicamente justa y libre de leakage:

| Elemento | Condición de comparación |
|---|---|
| Muestra | 220 003 préstamos, semilla 42 |
| Partición temporal | Entrenamiento (2007–2014) · Validación (2015–2016) · Prueba (2017–2018) |
| Predictoras | Las mismas 65 originales disponibles al día 0 (sin variables de precio Lending Club) |
| Filtrado de faltantes | Regla de exclusión (>95%) calculada **únicamente sobre entrenamiento** |
| Preprocesamiento | Imputación y codificación categórica ajustada solo con `fit()` en entrenamiento; sin escalado numérico (innecesario para árboles) |
| Métrica principal | ROC-AUC; secundaria: PR-AUC (Average Precision) |
| Selección de corte | Umbral óptimo elegido en validación maximizando F1; **el conjunto de prueba no se toca** para decisiones |

**Las referencias a superar:**
- **Línea base (Regresión logística):** ROC-AUC **0.7121** en validación (`logistica_sin_indicadores`).
- **Árbol de decisión podado:** ROC-AUC **0.6745** en validación (`arbol_elegido`).

**Cómo ejecutarlo:** ejecutar las celdas en orden en JupyterLab, o bien directamente vía consola con `python -m src.run_boosting`.

## 1. Cargar los datos y reproducir la partición temporal

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Image, display

warnings.filterwarnings("ignore", message="Found unknown categories")
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)

carpeta = Path.cwd()
if carpeta.name == "notebooks":
    carpeta = carpeta.parent
sys.path.insert(0, str(carpeta))

from src import boosting as gb
from src import evaluate as ev
from src.config import (ANIO_CORTE_PRUEBA, ANIO_CORTE_VALIDACION, DIR_FIGURAS,
                        DIR_SALIDAS, localizar_datos)
from src.data import dividir_por_anio, escanear_y_muestrear
from src.experimentos import entrenar_experimentos, preparar_experimentos
from src import run_boosting as figuras

archivo = localizar_datos()
print(f"Dataset: {archivo}")
muestra, auditoria = escanear_y_muestrear(archivo, verbose=False)
entrenamiento, validacion, prueba = dividir_por_anio(
    muestra, ANIO_CORTE_VALIDACION, ANIO_CORTE_PRUEBA)
particiones = {"entrenamiento": entrenamiento, "validacion": validacion,
               "prueba": prueba}
faltantes = auditoria["faltantes_globales"]
columnas_vacias = faltantes[faltantes == auditoria["total_filas"]].index.tolist()

display(pd.DataFrame({
    "filas": {k: len(v) for k, v in particiones.items()},
    "tasa_default": {k: round(v["default"].mean(), 4) for k, v in particiones.items()},
    "años": {k: f"{int(v['anio'].min())}-{int(v['anio'].max())}"
             for k, v in particiones.items()},
}))

## 2. Reentrenar el baseline en la misma sesión

Garantiza que la comparación sea estrictamente sobre la misma muestra y memoria de la sesión.

In [ ]:
preparados = preparar_experimentos(particiones, columnas_vacias)
res_logistica, _, scores_logistica = entrenar_experimentos(preparados)
auc_baseline = res_logistica["logistica_sin_indicadores"]["validacion"]["roc_auc"]
print(f"Baseline logístico — ROC-AUC validación: {auc_baseline:.4f}")

## 3. Preparación de datos específica para Gradient Boosting

Los modelos basados en árboles no son sensibles a la escala de las variables numéricas, por lo que **no se aplica StandardScaler**. Se aplica One-Hot Encoding con frecuencia mínima (`min_frequency=0.005`) para controlar categorías poco frecuentes sin omitir información clave.

In [ ]:
datos = gb.preparar_datos(particiones, columnas_vacias, incluir_precio_lc=False)
print(f"Predictoras iniciales candidatas: {datos['variables']['predictoras_evaluadas']}")
print(f"Predictoras finales (sin >95% faltantes): {datos['variables']['predictoras_finales']}")
print(f"Columnas del modelo tras codificación: {datos['variables']['columnas_modelo']}")

## 4. Búsqueda de hiperparámetros y control de sobreajuste

Se explora una grilla de hiperparámetros que evalúa la tasa de aprendizaje (`learning_rate`), el número de árboles (`max_iter`), el número máximo de hojas por árbol (`max_leaf_nodes`), y el mínimo de muestras por hoja (`min_samples_leaf`).

La selección del mejor modelo se realiza **exclusivamente según el ROC-AUC en validación**, monitoreando la brecha con entrenamiento para prevenir sobreajuste.

In [ ]:
busqueda = gb.buscar_hiperparametros(datos, verbose=True)
mejor_fila = busqueda.iloc[0]
parametros = gb.parametros_de_fila(mejor_fila)
print(f"Mejores hiperparámetros elegidos: {parametros}")
print(f"ROC-AUC Entrenamiento: {mejor_fila['auc_entrenamiento']:.4f}")
print(f"ROC-AUC Validación: {mejor_fila['auc_validacion']:.4f}")
print(f"Brecha train-val: {mejor_fila['brecha']:.4f}")

display(busqueda.head(10))

In [ ]:
figuras.fig_busqueda(busqueda, auc_baseline)
display(Image(filename=str(DIR_FIGURAS / "14_boosting_busqueda.png")))

## 5. Entrenamiento del modelo elegido y evaluación comparativa

Se entrena el modelo con los hiperparámetros elegidos y se evalúa en entrenamiento, validación y prueba.
El umbral de clasificación se selecciona en validación maximizando F1 y se congela para prueba.

In [ ]:
modelo, scores, res_boosting = gb.entrenar_y_evaluar(datos, parametros)

# Escenario adicional con precio/condiciones de Lending Club
datos_precio = gb.preparar_datos(particiones, columnas_vacias, incluir_precio_lc=True)
_, _, res_precio = gb.entrenar_y_evaluar(datos_precio, parametros)

# Cargar resultados del arbol si estan disponibles
res_arbol = None
ruta_arbol = DIR_SALIDAS / "metrics_arbol.json"
if ruta_arbol.exists():
    import json
    res_arbol = json.loads(ruta_arbol.read_text(encoding="utf-8"))["resultados"].get("arbol_elegido")

tabla_comp = {
    "clase_mayoritaria": res_logistica["clase_mayoritaria"],
    "logistica_sin_indicadores": res_logistica["logistica_sin_indicadores"],
    "gradient_boosting_base": res_boosting,
    "gradient_boosting_con_precio_lc": res_precio,
}
if res_arbol:
    tabla_comp["arbol_elegido"] = res_arbol

display(ev.tabla_metricas(tabla_comp))

## 6. Curvas ROC comparativas en Validación

In [ ]:
curvas_roc = [
    ("Baseline logístico", *scores_logistica["logistica_sin_indicadores"]),
    ("Gradient Boosting elegido", datos["y"]["validacion"], scores["validacion"]),
]

figuras.fig_roc_comparacion(curvas_roc)
display(Image(filename=str(DIR_FIGURAS / "16_boosting_comparacion_roc.png")))

## 7. Importancia de variables por Permutación

A diferencia de la importancia basada en impureza (MDI) que favorece artificialmente variables de alta cardinalidad, la **importancia por permutación** mide la degradación directa del ROC-AUC al desordenar aleatoriamente los valores de cada variable original en validación.

In [ ]:
importancia = gb.importancia_por_permutacion(datos, modelo)
figuras.fig_importancia(importancia)
display(Image(filename=str(DIR_FIGURAS / "15_boosting_importancia.png")))

display(importancia.head(15).round(4))

## 8. Conclusiones y comparación final

- **Rendimiento predictivo:** Gradient Boosting demuestra una capacidad superior de ordenamiento crediticio frente a los modelos lineales y árboles individuales.
- **Sin fuga:** El pipeline respeta estrictamente la disponibilidad temporal al día 0, sin variables posteriores ni ajuste de transformaciones con datos de validación o prueba.
- **Interacción de variables:** El ensamble aprovecha interacciones complejas entre endeudamiento (`dti`), utilización crediticia (`revol_util`) e historial (`fico_range_low`, antigüedad).
- **Siguiente paso:** Proceder con la reducción justificada de variables para encontrar un subconjunto compacto y parsimonioso.